In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlspchasiquiza01";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_finalproject CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_finalproject
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/catalog_finalprojectpc'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_finalproject.raw;
DROP SCHEMA IF EXISTS catalog_finalproject.bronze;
DROP SCHEMA IF EXISTS catalog_finalproject.silver;
DROP SCHEMA IF EXISTS catalog_finalproject.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_finalproject.raw;
CREATE SCHEMA IF NOT EXISTS catalog_finalproject.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_finalproject.silver;
CREATE SCHEMA IF NOT EXISTS catalog_finalproject.golden;

###Tablas Bronze

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.bronze.products_bz (
    product_id INT,
    product_name STRING,
    description STRING,
    category STRING,
    price DOUBLE,
    availability STRING,
    ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/products_bz";

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.bronze.customers_bz  (
    customer_id INT,
    name STRING,
    email STRING,
    country STRING,
    city STRING,
    phone STRING,
    created_date STRING,
    ingestion_date timestamp
) USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/customers_bz";


In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.bronze.orders_bz (
    order_id INT,
    customer_id INT,
    order_date STRING,
    channel STRING,
    branch_name STRING,
    state STRING,
    ingestion_date timestamp
) USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/orders_bz";

In [0]:
%sql

CREATE OR REPLACE TABLE catalog_finalproject.bronze.sales_bz (
    sale_id INT,
    order_id INT,
    product_id INT,
    quantity INT,
    price DOUBLE,
    total_amount DOUBLE,
    ingestion_date timestamp
) USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/sales_bz";

###Tablas Silver

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.silver.products_sz (
    product_id INT,
    product_name STRING,
    description STRING,
    category STRING,
    price DECIMAL(10,2),
    availability STRING
) USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/products_sz";


In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.silver.customers_sz (
    customer_id INT,
    name STRING,
    email STRING,
    country STRING,
    city STRING,
    phone STRING,
    created_date DATE
) USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/customers_sz";

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.silver.orders_sz (
    order_id INT,
    customer_id INT,
    order_date DATE,
    channel STRING,
    branch_name STRING,
    state STRING
) USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/orders_sz";


In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.silver.sales_sz (
    sale_id INT,
    order_id INT,
    product_id INT,
    quantity INT,
    price DECIMAL(10,2),
    total_amount DECIMAL(12,2)
) USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/sales_sz";

###Tablas Golden

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.golden.dim_date (
    date DATE,
    year INT,
    month INT,
    day INT,
    year_month STRING
) USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/dim_date";


In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.golden.dim_customer (
    customer_id INT,
    name STRING,
    email STRING,
    country STRING,
    city STRING,
    created_date DATE
) USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/dim_customer";

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.golden.dim_product (
    product_id INT,
    product_name STRING,
    category STRING,
    price DECIMAL(10,2),
    availability STRING
) USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/dim_product";

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.golden.dim_channel (
    channel STRING,
    branch_name STRING,
    state STRING
) USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/dim_channel";


In [0]:
%sql
CREATE OR REPLACE TABLE catalog_finalproject.golden.fact_sales (
    sale_id INT,
    order_id INT,
    customer_id INT,
    product_id INT,
    order_date DATE,
    channel STRING,
    branch_name STRING,
    state STRING,
    quantity INT,
    price DECIMAL(10,2),
    total_amount DECIMAL(12,2),
    discount_pct DECIMAL(5,4),
    net_amount DECIMAL(12,2),
    tax_amount DECIMAL(12,2),
    gross_amount DECIMAL(12,2),
    high_ticket_flag INT
) USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/fact_sales";